[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB00_Nivelacion_Datos_Machine_Learning.ipynb)

# NB0 — Nivelación: fundamentos prácticos para trabajar con datos en Machine Learning

**Correspondencia: Semana 2**

**Dataset:** `clientes_retail_video_limpieza.csv`


## 1. Cargar librerías


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split


## 2. Cargar el dataset

Suba el archivo **`clientes_retail_video_limpieza.csv`** entregado junto con este notebook.


In [ ]:
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
df = pd.read_csv(nombre_archivo)

print("Archivo cargado:", nombre_archivo)
df.head()


## 3. Inspeccionar los datos: `head()`, `shape`, `info()` y `describe()`


In [ ]:
display(df.head())

print("\nFilas y columnas:", df.shape)

print("\nInformación general:")
df.info()

print("\nResumen descriptivo:")
display(df.describe(include="all"))


## 4. Seleccionar filas y columnas

Operaciones básicas que se utilizarán durante todo el semestre:

- seleccionar una columna;
- seleccionar varias columnas;
- usar `loc`;
- usar `iloc`;
- aplicar filtros;
- ordenar resultados;
- revisar categorías;
- resumir información mediante `value_counts()` y `groupby()`.


In [ ]:
# Seleccionar una columna
display(df["monto_total"].head())

# Seleccionar varias columnas
display(df[["edad", "monto_total", "canal_preferido"]].head())

# loc: seleccionar por etiquetas/condiciones
display(df.loc[df["monto_total"] > 200000,
               ["id_cliente", "monto_total", "canal_preferido"]].head())

# iloc: seleccionar por posición
display(df.iloc[0:5, 0:4])

# Valores únicos y frecuencias
print("Canales registrados:")
print(df["canal_preferido"].unique())

print("\nFrecuencia por región:")
print(df["region"].value_counts())

# Ordenar
display(df.sort_values("monto_total", ascending=False).head())

# Resumen por grupo
display(
    df.groupby("canal_preferido", dropna=False)["monto_total"]
      .mean()
      .sort_values(ascending=False)
)


## 5. Identificar tipos de variables

En Machine Learning conviene distinguir, al menos:

- **identificadores**;
- **variables numéricas**;
- **variables categóricas**.

Un identificador como `id_cliente` no debe interpretarse como una característica cuantitativa del cliente.


In [ ]:
print(df.dtypes)

variables_numericas = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas = df.select_dtypes(include=["object"]).columns.tolist()

variables_numericas_analisis = [
    col for col in variables_numericas
    if col != "id_cliente"
]

print("\nVariables numéricas:", variables_numericas)
print("Variables numéricas para análisis:", variables_numericas_analisis)
print("Variables categóricas:", variables_categoricas)


## 6. Detectar valores faltantes

Los valores faltantes pueden afectar cálculos, visualizaciones y algoritmos posteriores.


In [ ]:
faltantes = df.isnull().sum().sort_values(ascending=False)

print("Valores faltantes por columna:")
print(faltantes)

print("\nTotal de valores faltantes:", int(faltantes.sum()))


## 7. Imputar o eliminar valores faltantes

En este ejemplo se utilizará la **mediana** para imputar valores faltantes en `edad` y `monto_total`.

La estrategia adecuada depende del problema, del tipo de variable y de la cantidad de datos faltantes.


In [ ]:
df_trabajo = df.copy()

df_trabajo["edad"] = df_trabajo["edad"].fillna(
    df_trabajo["edad"].median()
)

df_trabajo["monto_total"] = df_trabajo["monto_total"].fillna(
    df_trabajo["monto_total"].median()
)

print("Valores faltantes después de imputar:")
print(df_trabajo.isnull().sum())


## 8. Detectar y eliminar duplicados


In [ ]:
print("Duplicados detectados:", df_trabajo.duplicated().sum())

df_trabajo = df_trabajo.drop_duplicates().copy()

print("Duplicados después de limpiar:", df_trabajo.duplicated().sum())
print("Dimensión actual:", df_trabajo.shape)


## 9. Corregir categorías inconsistentes

Una misma categoría puede aparecer escrita de distintas formas. Por ejemplo:

`Online`, ` online `, `onl`, `web`.

Antes de analizar o codificar una variable categórica, conviene estandarizar sus valores.


In [ ]:
print("Antes:")
print(df_trabajo["canal_preferido"].value_counts(dropna=False))

df_trabajo["canal_preferido"] = (
    df_trabajo["canal_preferido"]
    .str.strip()
    .str.lower()
    .replace({
        "online": "Online",
        "onl": "Online",
        "web": "Online",
        "tienda": "Tienda",
        "presencial": "Tienda"
    })
)

df_trabajo["categoria_frecuente"] = (
    df_trabajo["categoria_frecuente"]
    .str.strip()
    .str.title()
)

print("\nDespués:")
print(df_trabajo["canal_preferido"].value_counts())
print("\nCategorías frecuentes:")
print(df_trabajo["categoria_frecuente"].value_counts())


## 10. Detectar valores fuera de rango

Un valor puede ser numérico y, sin embargo, no ser razonable para el problema.

En este dataset:

- se considerará válida una edad entre **18 y 80 años**;
- `frecuencia_compra` no puede ser negativa.


In [ ]:
print("Edades fuera de rango:")
display(
    df_trabajo.loc[
        (df_trabajo["edad"] < 18) | (df_trabajo["edad"] > 80),
        ["id_cliente", "edad"]
    ]
)

print("Frecuencias negativas:")
display(
    df_trabajo.loc[
        df_trabajo["frecuencia_compra"] < 0,
        ["id_cliente", "frecuencia_compra"]
    ]
)

mediana_edad = df_trabajo.loc[
    df_trabajo["edad"].between(18, 80),
    "edad"
].median()

df_trabajo.loc[
    ~df_trabajo["edad"].between(18, 80),
    "edad"
] = mediana_edad

df_trabajo.loc[
    df_trabajo["frecuencia_compra"] < 0,
    "frecuencia_compra"
] = 0

print("Correcciones aplicadas.")


## 11. Codificar variables categóricas

Muchos algoritmos necesitan entradas numéricas. Una alternativa es **one-hot encoding**, que crea columnas binarias para representar cada categoría.


In [ ]:
variables_categoricas = df_trabajo.select_dtypes(
    include=["object"]
).columns.tolist()

df_codificado = pd.get_dummies(
    df_trabajo,
    columns=variables_categoricas,
    drop_first=False,
    dtype=int
)

print("Columnas antes:", df_trabajo.shape[1])
print("Columnas después:", df_codificado.shape[1])

display(df_codificado.head())


## 12. Normalizar y estandarizar

### Normalización
Transforma los valores a un rango común, normalmente **0–1**.

### Estandarización
Transforma una variable para que tenga aproximadamente:

- media = 0;
- desviación estándar = 1.

`id_cliente` se excluye porque es un identificador.


In [ ]:
variables_numericas = df_trabajo.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

variables_numericas_analisis = [
    col for col in variables_numericas
    if col != "id_cliente"
]

# Normalización
normalizador = MinMaxScaler()

df_normalizado = df_codificado.copy()
df_normalizado[variables_numericas_analisis] = normalizador.fit_transform(
    df_normalizado[variables_numericas_analisis]
)

# Estandarización
estandarizador = StandardScaler()

df_estandarizado = df_codificado.copy()
df_estandarizado[variables_numericas_analisis] = estandarizador.fit_transform(
    df_estandarizado[variables_numericas_analisis]
)

print("Normalizado:")
display(df_normalizado[variables_numericas_analisis].head())

print("Estandarizado:")
display(df_estandarizado[variables_numericas_analisis].head())


## 13. Visualización básica

Las visualizaciones permiten reconocer distribuciones, diferencias entre grupos y posibles anomalías.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_trabajo["monto_total"].dropna(), bins=12)
plt.title("Distribución de monto_total")
plt.xlabel("Monto total")
plt.ylabel("Frecuencia")
plt.show()

promedio_canal = (
    df_trabajo.groupby("canal_preferido")["monto_total"]
    .mean()
)

plt.figure(figsize=(6, 4))
promedio_canal.plot(kind="bar")
plt.title("Monto total promedio por canal")
plt.xlabel("Canal preferido")
plt.ylabel("Monto promedio")
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(
    df_trabajo["frecuencia_compra"],
    df_trabajo["monto_total"],
    alpha=0.7
)
plt.title("Frecuencia de compra vs. monto total")
plt.xlabel("Frecuencia de compra")
plt.ylabel("Monto total")
plt.show()


## 14. Separar datos en entrenamiento y prueba

Una práctica fundamental en Machine Learning es reservar datos que no participen del entrenamiento.

En este ejemplo se utilizará una división:

- **70% entrenamiento**;
- **30% prueba**.

También se muestra una partición estratificada utilizando `canal_preferido` como referencia pedagógica.


In [ ]:
train_df, test_df = train_test_split(
    df_estandarizado,
    test_size=0.30,
    random_state=42
)

print("Entrenamiento:", train_df.shape)
print("Prueba:", test_df.shape)

# Ejemplo de estratificación
train_strat, test_strat = train_test_split(
    df_estandarizado,
    test_size=0.30,
    random_state=42,
    stratify=df_trabajo["canal_preferido"]
)

print("\nDistribución original:")
print(df_trabajo["canal_preferido"].value_counts(normalize=True))

print("\nDistribución entrenamiento estratificado:")
print(
    df_trabajo.loc[train_strat.index, "canal_preferido"]
    .value_counts(normalize=True)
)

print("\nDistribución prueba estratificada:")
print(
    df_trabajo.loc[test_strat.index, "canal_preferido"]
    .value_counts(normalize=True)
)


## 15. Exportar el resultado

Al finalizar una preparación de datos conviene guardar el resultado y mantener trazabilidad de las transformaciones realizadas.


In [ ]:
# Validación final
print("Valores faltantes:", df_estandarizado.isnull().sum().sum())
print("Duplicados:", df_estandarizado.duplicated().sum())
print("Dimensión final:", df_estandarizado.shape)

# Exportar
archivo_salida = "clientes_retail_preparado_NB0.csv"
df_estandarizado.to_csv(archivo_salida, index=False)

print("\nArchivo generado:", archivo_salida)

# Descargar desde Colab
files.download(archivo_salida)


## Checklist de nivelación

Al finalizar NB0, el estudiante debería ser capaz de:

- cargar un CSV en Google Colab;
- inspeccionar un DataFrame;
- seleccionar y filtrar datos;
- distinguir variables numéricas, categóricas e identificadores;
- detectar e imputar valores faltantes;
- detectar y eliminar duplicados;
- corregir categorías inconsistentes;
- identificar valores fuera de rango;
- aplicar one-hot encoding;
- normalizar y estandarizar variables;
- realizar visualizaciones básicas;
- separar datos en entrenamiento y prueba;
- exportar un dataset preparado.
